# AIDev v3 — G0 historical cohort check (China Task Innovation Dynamics project)

**操作：点击 Runtime → Run all（运行所有单元格）。** 仅需免费 CPU 环境，不用上传、下载或修改文件。结果直接显示在最后一个单元格；把最后的完整输出复制回 ChatGPT。

Data: official [AIDev v3](https://huggingface.co/datasets/hao-li/AIDev/tree/v3); benchmark: Hindi et al. (2026) [IEEE Access](https://doi.org/10.1109/ACCESS.2026.3696573).

Scope: only `pull_request.parquet` (~16.9MB) and `repository.parquet` (~0.17MB), same v3 commit, repo stars ≥500, five predefined agent labels, PR created before Aug 2, 2025 (UTC). Published comparison counts: 12,433 PRs, 1,495 repos, 11,100 closed, 4,283 rejected. **These are not filters or research results.** No regression, model, or raw PR data export. A mismatch is an important finding requiring provenance audit, not a reason to change research cutoffs.

Current-public metadata/source vintage versus historical paper's as-of date, outcomes, repo popularity, and source agent attribution must be kept distinct. This script performs a low-cost G0 source feasibility check, **not a G0 PASS**.

## R304 — 同一 v3 数据版本的来源差异核验
点击 **运行时 → 全部运行**。新增最后一个单元格仅读取较小的 `all_repository.parquet`，比较其与热门仓库表在 >=500 星时的仓库覆盖情况。**不会下载大型完整 PR 表、不会运行回归。**


## R306 — 正式输入源的限定范围核验
在 **Runtime → Run all** 运行全部单元格。第五个单元格使用同一不可变 AIDev v3 提交的 `all_pull_request.parquet`，通过官方 Hugging Face 远程文件系统读取必要列、分批计算 17 个资格差异仓库及原有仓库的样本和 PR ID 一致性；它不主动下载完整大文件，也不保存任何 PR 内容或运行回归。若远程按列读取失败，将报错并停止。把最后的 R306 输出发给我即可。


### R307 执行缺陷修复
第 5 个代码单元格现已**完全独立运行**：自动检查依赖、强制使用固定 v3 commit、校验三个小文件 SHA-256、重建原有 POP 统计，再远程扫描完整 PR 表的必要列。不必运行前四格。若运行失败，不会改筛选参数，也不会自动下载完整大型 PR 文件。


### R308 — 最后 6 条 PR 的来源定位（不改变样本规则）
直接运行第 5 个单元格。新增输出检查恰好 500 星的仓库、唯一发生的星数差异，以及是否有单个仓库符合 6 条 PR / 5 条关闭 / 2 条拒绝的特征。它**不删除任何仓库或 PR**，仅诊断余下差异；结果分为 R308 诊断和原有 R306 统计两段。


In [ ]:
%pip -q install "huggingface_hub>=0.29" "pandas>=2" "pyarrow>=15"

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

DS, TAG = "hao-li/AIDev", "v3"
FILENAMES = ("pull_request.parquet", "repository.parquet")
AGENTS = ("OpenAI_Codex", "Devin", "Copilot", "Cursor", "Claude_Code")
STAR_MIN = 500
END = pd.Timestamp("2025-08-02T00:00:00Z")
MODEL_START = pd.Timestamp("2024-12-01T00:00:00Z")
MODEL_END = pd.Timestamp("2025-08-01T00:00:00Z")
PUBLISHED = {"prs":12433, "repos":1495, "closed":11100, "rejected":4283}
LIMIT = 60 * 1024 * 1024

# Resolve the published historical tag to an immutable revision.
info = HfApi().repo_info(repo_id=DS, repo_type="dataset", revision=TAG, files_metadata=True)
revision = info.sha
if not revision or len(revision) != 40:
    raise RuntimeError("STOP: Cannot verify immutable v3 revision.")
inventory = {obj.rfilename: obj for obj in info.siblings or []}
for name in FILENAMES:
    if name not in inventory:
        raise RuntimeError(f"STOP: Missing {name} in v3. Do not use latest/main!")
    nbytes = getattr(inventory[name], "size", None)
    if nbytes is not None and nbytes > LIMIT:
        raise RuntimeError(f"STOP: {name} unexpectedly large: {nbytes} bytes")
print("Pinned official dataset:", DS, "tag:", TAG, "commit:", revision)

paths, metadata = {}, {}
for name in FILENAMES:
    file = Path(hf_hub_download(repo_id=DS, repo_type="dataset", filename=name, revision=revision))
    if file.stat().st_size > LIMIT:
        raise RuntimeError(f"STOP: downloaded file {name} exceeded size limit")
    digest = hashlib.sha256()
    with file.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    parquet = pq.ParquetFile(file)
    metadata[name] = {
        "bytes":file.stat().st_size, "sha256":digest.hexdigest(),
        "rows_footer":parquet.metadata.num_rows,
        "columns":parquet.schema.names,
    }
    paths[name] = file
print("\nSOURCE INTEGRITY (original bytes):")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

prs = pd.read_parquet(paths["pull_request.parquet"])
repos = pd.read_parquet(paths["repository.parquet"])
need_pr = {"id","repo_id","agent","created_at","closed_at","merged_at","state"}
need_repo = {"id","stars"}
missing_pr = sorted(need_pr - set(prs.columns))
missing_repo = sorted(need_repo - set(repos.columns))
if missing_pr or missing_repo:
    raise RuntimeError(f"STOP source schema mismatch. PR missing={missing_pr}; repo missing={missing_repo}")
for tab, key in ((prs,"id"),(repos,"id")):
    if tab[key].isna().any() or tab[key].duplicated().any():
        raise RuntimeError(f"STOP: missing/duplicated {key}; no automatic deduplication")

stars = pd.to_numeric(repos["stars"], errors="coerce")
pop_ids = set(repos.loc[stars >= STAR_MIN, "id"])
p = prs.loc[prs["repo_id"].isin(pop_ids) & prs["agent"].astype(str).isin(AGENTS)].copy()
created = pd.to_datetime(p["created_at"], utc=True, errors="coerce")
p = p.loc[created.lt(END)].copy()
created = pd.to_datetime(p["created_at"], utc=True, errors="coerce")
closed = pd.to_datetime(p["closed_at"], utc=True, errors="coerce")
merged = pd.to_datetime(p["merged_at"], utc=True, errors="coerce")
state_closed = p["state"].astype(str).str.lower().eq("closed")
closed_snapshot = state_closed | merged.notna()
rejected_snapshot = closed_snapshot & merged.isna()
closed_asof = closed.lt(END) | merged.lt(END)
merged_asof = merged.lt(END)
rejected_asof = closed_asof & ~merged_asof
in_model_months = created.ge(MODEL_START) & created.lt(MODEL_END)

observed = {
  "prs":int(len(p)),
  "repos":int(p["repo_id"].nunique()),
  "closed_snapshot":int(closed_snapshot.sum()),
  "rejected_snapshot":int(rejected_snapshot.sum()),
  "closed_asof":int(closed_asof.sum()),
  "rejected_asof":int(rejected_asof.sum()),
  "model_month_prs_dec2024_jul2025":int(in_model_months.sum()),
  "model_month_closed_snapshot":int((in_model_months & closed_snapshot).sum()),
  "model_month_closed_asof":int((in_model_months & closed_asof).sum()),
  "raw_pop_pr_rows":int(len(prs)),
  "raw_pop_repo_rows":int(len(repos)),
  "missing_repo_fk":int((~prs["repo_id"].isin(repos["id"])).sum()),
  "missing_stars":int(stars.isna().sum()),
  "missing_created_cohort":int(created.isna().sum()),
  "later_closures_in_snapshot":int((closed_snapshot & ~closed_asof).sum()),
  "agent_counts":{str(k):int(v) for k,v in p["agent"].value_counts().items()},
}

print("\n=== OBSERVED SOURCE COUNTS (not paper replication proof) ===")
print(json.dumps({"observed":observed,"published_comparison_not_filters":PUBLISHED},
                 indent=2, ensure_ascii=False))
print("\nCOMPARISON: total PR difference =",observed["prs"]-PUBLISHED["prs"],
      "; repo difference =",observed["repos"]-PUBLISHED["repos"])
print("RESEARCH STOP RULE: matching counts alone does not verify original PR IDs,")
print("historical star snapshot, agent attribution, outcome as-of timing, or regression.")
print("G0 source check only. No fitted model, no G0/G1 PASS.")


In [ ]:
# R304: only a small same-commit repository-universe comparison; no large PR table
from pathlib import Path
import json, hashlib
from huggingface_hub import hf_hub_download
import pandas as pd
name="all_repository.parquet"
meta=inventory.get(name)
if meta is None: raise RuntimeError("STOP: all_repository.parquet not in pinned v3")
if getattr(meta,"size",None) is not None and meta.size>20_000_000: raise RuntimeError("STOP: unexpectedly large repository file")
f=Path(hf_hub_download(repo_id=DS,repo_type="dataset",filename=name,revision=revision))
if f.stat().st_size>20_000_000: raise RuntimeError("STOP: unexpected byte size")
allrepo=pd.read_parquet(f)
if not {"id","stars"}.issubset(allrepo): raise RuntimeError("STOP: different repository schema")
if allrepo.id.isna().any() or allrepo.id.duplicated().any(): raise RuntimeError("STOP: nonunique all repository id")
a=pd.to_numeric(allrepo.stars,errors="coerce")
pop_ge500=set(repos.loc[stars>=500,"id"])
all_ge500=set(allrepo.loc[a>=500,"id"])
joined=repos[["id","stars"]].merge(allrepo[["id","stars"]],on="id",suffixes=("_pop","_all"))
differ=(pd.to_numeric(joined.stars_pop,errors="coerce")!=pd.to_numeric(joined.stars_all,errors="coerce"))
output={
"commit":revision,"all_repo_bytes":f.stat().st_size,
"all_repo_sha256":hashlib.sha256(f.read_bytes()).hexdigest(),
"all_repo_rows":int(len(allrepo)),"pop_repo_rows":int(len(repos)),
"all_ge500_repo_count":len(all_ge500),
"pop_ge500_repo_count":len(pop_ge500),
"eligible_all_missing_in_pop":len(all_ge500-pop_ge500),
"eligible_pop_missing_in_all":len(pop_ge500-all_ge500),
"overlap_repo_ids":int(len(joined)),"overlap_star_disagreement":int(differ.sum()),
"cohort_repo_ids_not_in_all_ge500":len(set(p.repo_id)-all_ge500),
"prior_pop_sample":{"prs":observed["prs"],"repos":observed["repos"]},
"published_target":{"prs":12433,"repos":1495}
}
print("=== R304 SAME-v3 ALL vs POP REPOSITORY AUDIT ===")
print(json.dumps(output,ensure_ascii=False,indent=2))
print("NO full PR table retrieved; differences do not prove published error.")


In [ ]:
# R307: independent bootstrap; this cell runs even after Colab kernel reset.
import importlib.util, subprocess, sys
for mod, spec in [("huggingface_hub","huggingface_hub>=0.29"),
                  ("pyarrow","pyarrow>=15"),("pandas","pandas>=2"),
                  ("fsspec","fsspec>=2024.3")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",spec])
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import pandas as pd, hashlib, json, pyarrow.parquet as pq
DS, TAG = "hao-li/AIDev", "v3"
EXPECTED_REVISION = "68ed5f4b80d27a9e057fc57567f38bd322ac73ec"
revision = HfApi().repo_info(repo_id=DS,repo_type="dataset",
                             revision=TAG,files_metadata=True)
if revision.sha != EXPECTED_REVISION:
    raise RuntimeError("STOP: v3 tag no longer resolves to the verified historical commit")
inventory = {x.rfilename:x for x in revision.siblings or []}
revision = EXPECTED_REVISION
checks = {
    "pull_request.parquet": ("08f520b5ef36b6281f82090db09ac14aefc3534ee113886e7bc85131fd8d214c", 60_000_000),
    "repository.parquet": ("5b76717c39be1cada097d680aa0fcecd03611aa52ae05ef7f1d1e0e1e8b34192", 5_000_000),
    "all_repository.parquet": ("682a612d420420b1c738930332c78eae46268fc3bee7de12ca2f49da657741b1", 20_000_000)
}
tables={}
for name,(expected_hash,max_bytes) in checks.items():
    x=inventory.get(name)
    if x is None or (x.size is not None and x.size>max_bytes):
        raise RuntimeError("STOP: missing or oversized pinned source "+name)
    filename=Path(hf_hub_download(repo_id=DS,repo_type="dataset",
                                  filename=name,revision=revision))
    if filename.stat().st_size>max_bytes:
        raise RuntimeError("STOP: downloaded byte limit exceeded "+name)
    digest=hashlib.sha256(filename.read_bytes()).hexdigest()
    if digest!=expected_hash:
        raise RuntimeError("STOP: original source SHA-256 mismatch "+name)
    tables[name]=filename
repos=pd.read_parquet(tables["repository.parquet"],columns=["id","stars"])
allrepo=pd.read_parquet(tables["all_repository.parquet"],columns=["id","stars"])
prs=pd.read_parquet(tables["pull_request.parquet"],
                    columns=["id","repo_id","agent","created_at","state","closed_at","merged_at"])
for tbl in [repos,allrepo,prs]:
    if tbl["id"].isna().any() or tbl["id"].duplicated().any():
        raise RuntimeError("STOP: source identifier not unique")
STAR_MIN=500
AGENTS=("OpenAI_Codex","Devin","Copilot","Cursor","Claude_Code")
END=pd.Timestamp("2025-08-02T00:00:00Z")
MODEL_START=pd.Timestamp("2024-12-01T00:00:00Z")
MODEL_END=pd.Timestamp("2025-08-01T00:00:00Z")
PUBLISHED={"prs":12433,"repos":1495,"closed":11100,"rejected":4283}
stars=pd.to_numeric(repos["stars"],errors="coerce")
allstars=pd.to_numeric(allrepo["stars"],errors="coerce")
pop_ge500=set(repos.loc[stars.ge(STAR_MIN),"id"])
all_ge500=set(allrepo.loc[allstars.ge(STAR_MIN),"id"])
if len(all_ge500-pop_ge500)!=17 or len(pop_ge500-all_ge500)!=0:
    raise RuntimeError("STOP: predeclared 17-repo eligibility difference not reproduced")
dates=pd.to_datetime(prs["created_at"],utc=True,errors="coerce")
mask=prs["repo_id"].isin(pop_ge500)&prs["agent"].astype(str).isin(AGENTS)&dates.lt(END)
p=prs.loc[mask].copy()
omerged=pd.to_datetime(p["merged_at"],utc=True,errors="coerce")
oclosed=p["state"].astype(str).str.lower().eq("closed")|omerged.notna()
orejected=oclosed & omerged.isna()
observed={"prs":int(len(p)),"repos":int(p["repo_id"].nunique()),
          "closed_snapshot":int(oclosed.sum()),"rejected_snapshot":int(orejected.sum())}
if observed!={"prs":12373,"repos":1479,"closed_snapshot":11048,"rejected_snapshot":4274}:
    raise RuntimeError("STOP: previous verified POP cohort not reconstructed")
print("R307 standalone bootstrap PASS: same v3 commit, verified 3 source hashes, 17-repo difference, POP cohort counts")
# Continue with unchanged R306 remote ALL PR projected-column scan below.

# R306: pinned-v3 remote Parquet column/batch scan, no full-file download
from huggingface_hub import HfFileSystem
from collections import Counter
import pyarrow.parquet as pq
import pandas as pd, json
if revision!="68ed5f4b80d27a9e057fc57567f38bd322ac73ec":
    raise RuntimeError("STOP: historical v3 source revision changed")
fn="all_pull_request.parquet"
obj=inventory.get(fn)
if obj is None: raise RuntimeError("STOP: expected original ALL PR source is missing")
if obj.size is not None and obj.size>400_000_000:
    raise RuntimeError("STOP: original ALL PR source exceeds predeclared 400 MB guard")
target=all_ge500-pop_ge500
if len(target)!=17: raise RuntimeError("STOP: eligibility frame changed")
cols=["id","repo_id","agent","created_at","state","closed_at","merged_at"]
results={k:{"ids":set(),"repo_ids":set(),"closed":0,"rejected":0,"closed_asof":0,"rejected_asof":0,
            "closed_in_model_months":0,"rejected_in_model_months":0}
         for k in ("extra17","pop1479","all")}
scan_rows=0
repo_audit={}
def accumulate(df,label,created):
    if df.empty: return
    x=results[label]
    ids=set(df["id"])
    if df["id"].isna().any() or len(ids)!=len(df) or ids & x["ids"]:
        raise RuntimeError("STOP: duplicate or null selected PR IDs")
    x["ids"].update(ids)
    x["repo_ids"].update(df["repo_id"])
    if label=="all":
        for rid,g in df.groupby("repo_id",sort=False):
            row=repo_audit.setdefault(rid,{"prs":0,"closed":0,"rejected":0})
            gm=pd.to_datetime(g["merged_at"],utc=True,errors="coerce")
            gc=g["state"].astype(str).str.lower().eq("closed")|gm.notna()
            row["prs"]+=len(g)
            row["closed"]+=int(gc.sum())
            row["rejected"]+=int((gc&gm.isna()).sum())
    merged=pd.to_datetime(df["merged_at"],utc=True,errors="coerce")
    closed=pd.to_datetime(df["closed_at"],utc=True,errors="coerce")
    cm=df["state"].astype(str).str.lower().eq("closed") | merged.notna()
    rj=cm & merged.isna()
    ca=closed.lt(END) | merged.lt(END)
    ra=ca & ~merged.lt(END)
    in_month=created.ge(MODEL_START)&created.lt(MODEL_END)
    x["closed"]+=int(cm.sum());x["rejected"]+=int(rj.sum())
    x["closed_asof"]+=int(ca.sum());x["rejected_asof"]+=int(ra.sum())
    x["closed_in_model_months"]+=int((cm&in_month).sum())
    x["rejected_in_model_months"]+=int((rj&in_month).sum())
remote=f"datasets/{DS}@{revision}/{fn}"
fs=HfFileSystem()
try:
    with fs.open(remote,"rb",block_size=4_194_304) as reader:
        pf=pq.ParquetFile(reader)
        if not set(cols).issubset(set(pf.schema_arrow.names)):
            raise RuntimeError("STOP: ALL PR schema mismatch; do not infer counts")
        all_footer_rows=int(pf.metadata.num_rows)
        for batch in pf.iter_batches(batch_size=16384,columns=cols,use_threads=False):
            df=batch.to_pandas()
            scan_rows+=len(df)
            mask=df["repo_id"].isin(all_ge500)&df["agent"].astype(str).isin(AGENTS)
            df=df.loc[mask].copy()
            if df.empty: continue
            dates=pd.to_datetime(df["created_at"],utc=True,errors="coerce")
            df=df.loc[dates.lt(END)].copy()
            dates=pd.to_datetime(df["created_at"],utc=True,errors="coerce")
            if df.empty: continue
            accumulate(df,"all",dates)
            em=df["repo_id"].isin(target)
            accumulate(df.loc[em],"extra17",dates.loc[em])
            accumulate(df.loc[~em],"pop1479",dates.loc[~em])
except Exception as e:
    raise RuntimeError("STOP: pinned remote Parquet scan failed; full PR file NOT downloaded. "+
                       type(e).__name__+": "+str(e)[:350]) from e
if scan_rows!=all_footer_rows: raise RuntimeError("STOP: incomplete remote PR table scan")
old=set(p["id"])
shared=results["pop1479"]["ids"]
def public(k):
    x=results[k]
    return {"prs":len(x["ids"]),"repos_with_pr":len(x["repo_ids"]),
            **{a:x[a] for a in ("closed","rejected","closed_asof","rejected_asof",
                              "closed_in_model_months","rejected_in_model_months")}}
summary={"revision":revision,"source":fn,"official_size_bytes":obj.size,
         "rows_scanned":scan_rows,"file_sha256":"NOT_COMPUTED_remote_partial_column_read",
         "distinct_eligible_added_repo_ids":len(target),
         "new_only":public("extra17"),"existing_in_all":public("pop1479"),
         "all_eligible_full_source":public("all"),
         "old_pop_reproduced":{"prs":observed["prs"],"repos":observed["repos"],
                               "closed":observed["closed_snapshot"],"rejected":observed["rejected_snapshot"]},
         "same_existing_sample_ids":len(shared)==len(old) and shared==old,
         "all_missing_prior_pop_ids":len(old-shared),
         "additional_existing_all_ids_not_in_pop":len(shared-old),
         "published_comparison_not_filters":PUBLISHED}
# R308 predeclared, complete repo-level signature and boundary inspection: NO rows deleted.
allstar=dict(zip(allrepo["id"],allstars))
popstar=dict(zip(repos["id"],stars))
star_disagreement=[]
for rid in set(popstar)&set(allstar):
    if popstar[rid]!=allstar[rid]:
        star_disagreement.append({"repo_id":str(rid),"stars_pop":float(popstar[rid]),
                                 "stars_all":float(allstar[rid])})
star_disagreement.sort(key=lambda x:x["repo_id"])
eq500={rid:v for rid,v in repo_audit.items() if allstar.get(rid)==500}
sig=[(rid,v) for rid,v in repo_audit.items()
     if (v["prs"],v["closed"],v["rejected"])==(6,5,2)]
sig.sort(key=lambda x:str(x[0]))
star500sig=[rid for rid,v in sig if allstar.get(rid)==500]
def total(rows,key): return sum(v[key] for v in rows.values())
audit={
    "exactly_500_star_repo_with_pr":len(eq500),
    "exactly_500_star_prs":total(eq500,"prs"),
    "exactly_500_star_closed":total(eq500,"closed"),
    "exactly_500_star_rejected":total(eq500,"rejected"),
    "overlap_star_disagreement":star_disagreement,
    "repo_6prs_5closed_2rejected_candidate_count":len(sig),
    "repo_6_5_2_candidate_500_star_ids":[str(x) for x in star500sig],
    "candidate_repo_summaries_at_most_20":[{"repo_id":str(rid),
        "stars_in_all":float(allstar[rid]),**v} for rid,v in sig[:20]],
    "published_minus_same_v3_all":{"prs":PUBLISHED["prs"]-public("all")["prs"],
        "repos":PUBLISHED["repos"]-public("all")["repos_with_pr"],
        "closed":PUBLISHED["closed"]-public("all")["closed"],
        "rejected":PUBLISHED["rejected"]-public("all")["rejected"]}
}
print("=== R308 EXACT 500-STAR AND SIX-PR RECONCILIATION AUDIT ===")
print(json.dumps(audit,indent=2,ensure_ascii=False))
print("No source row excluded, no changed threshold, no claim of paper replication.")

print("=== R306 PINNED V3 FULL-SOURCE PR COHORT COMPARISON ===")
print(json.dumps(summary,indent=2,ensure_ascii=False))
print("Do not infer exact publication replication from counts alone; no fitted models; G0 remains REDESIGN.")
